# 10. UPDATE & DELETE

*Module: 3 — DML · Companion summary: `lectures/10-update-and-delete.md` · SQL engine: SQLite (in-memory)*

## What you will study

- write UPDATE … SET … WHERE and DELETE FROM … WHERE
- run the golden safety rule: preview the WHERE as a SELECT first
- see what a missing WHERE does — safely, inside a transaction

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 The AUTHOR table + step 1: always preview

In [ ]:
q("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT, firstname TEXT, email TEXT, city TEXT, country TEXT)")
q("""
INSERT INTO author VALUES
    ('A1','Chong','Raul','RFC@ca.ibm.com','Toronto','CA'),
    ('A2','Ahuja','Rav','RA@ca.ibm.com','Markham','CA'),
    ('A3','Katta','Lakshmi','LK@in.ibm.com','Bengaluru','IN')
""")
q("SELECT * FROM author WHERE author_id = 'A2'")   # the rows UPDATE will touch


## 🧪 UPDATE — change firstname/lastname of author 'A2' (course example)

In [ ]:
q("UPDATE author SET lastname = 'Katta', firstname = 'Lakshmi' WHERE author_id = 'A2'")
q("SELECT * FROM author")

## 🧪 DELETE — remove the rows for 'A2' and 'A3' (course example)

In [ ]:
q("SELECT * FROM author WHERE author_id IN ('A2','A3')")   # preview first
q("DELETE FROM author WHERE author_id IN ('A2','A3')")
q("SELECT * FROM author")

## 🧪 The missing-WHERE disaster — demonstrated *safely*

A WHERE-less UPDATE hits **every row**. Inside a transaction we can watch it and Roll Back:

In [ ]:
q("INSERT INTO author VALUES ('A2','Ahuja','Rav','RA@ca.ibm.com','Markham','CA')")   # A1 never left
conn.commit()                                  # close the implicit write-txn before an explicit BEGIN
q("BEGIN")                                     # open a transaction (lecture 29)
q("UPDATE author SET firstname = 'Oops', lastname = 'Oops'")   # no WHERE!
q("SELECT * FROM author")                      # every row changed
conn.rollback()                                # undo everything
print('--- after rollback ---')
q("SELECT * FROM author")

## 💡 Tips & tricks (the golden safety rules)

1. Write the **WHERE clause first**, before the UPDATE/DELETE body.
2. Run the same WHERE as a **SELECT** to preview the target rows.
3. Only then convert to UPDATE/DELETE — preferably inside a transaction.
4. DELETE (DML) removes rows; TRUNCATE (DDL, nb 06) empties a table faster.

## 🎯 Your turn

1. UPDATE all authors' city — but *safely*: begin a transaction, view the damage, roll back.
2. DELETE an author by id, previewed and verified.

In [ ]:
# 💡 Solution
q("BEGIN")
q("UPDATE author SET city = 'Nowhere'")
q("SELECT * FROM author")
conn.rollback()
q("SELECT * FROM author")

q("SELECT * FROM author WHERE author_id = 'A2'")     # preview
q("DELETE FROM author WHERE author_id = 'A2'")
q("SELECT * FROM author")